# Macro/Liquidity Research Workflow

Objetivo: responder, para el cliente del IPS, en que regimen macroeconomico y de liquidez se encuentra Estados Unidos, que regimen se espera en el futuro cercano y que tipos de activos tienen mejor asimetria riesgo-retorno bajo esa combinacion.

In [ ]:
import pandas as pd
import numpy as np

from src.client_policy import mexican_moderate_aggressive_growth_policy

from src.research.macro_liquidity import (
    ForecastScenario,
    LocalSeriesProvider,
    MacroLiquidityResearch,
    ProviderConfig,
    RegimeForecast,
    build_asset_class_view,
    default_us_macro_liquidity_catalog,
)

from src.strategy import build_selection_context

## 1. Revisar disponibilidad de APIs

Las credenciales deben vivir fuera del repo, como variables de entorno. Esta celda solo revisa que providers estan listos.

In [ ]:
provider_config = ProviderConfig()
provider_config.availability_report()

## 2. Cargar series macro/liquidez

Para un primer research reproducible, usa datos cacheados en formato largo: `date`, `series`, `value`. Luego se puede reemplazar `LocalSeriesProvider` por providers API.

In [ ]:
catalog = default_us_macro_liquidity_catalog()

# Reemplazar por tu archivo cuando exista:
# series_frame = pd.read_csv('../data/interim/macro_liquidity_series.csv')

dates = pd.date_range('2024-01-31', periods=8, freq='ME')
series_frame = pd.DataFrame([
    {'date': date, 'series': series, 'value': value}
    for series, values in {
        'real_gdp': [100, 101, 102, 103, 104, 105, 106, 107],
        'cpi': [110, 111, 112, 112, 111, 111, 110, 110],
        'unemployment_rate': [4.2, 4.2, 4.1, 4.1, 4.0, 4.0, 3.9, 3.9],
        'reserve_balances': [3.0, 3.1, 3.2, 3.2, 3.1, 3.0, 2.9, 2.8],
        'tga': [700, 720, 760, 790, 820, 860, 890, 930],
        'hy_oas': [3.5, 3.6, 3.8, 4.0, 4.2, 4.4, 4.7, 4.9],
    }.items()
    for date, value in zip(dates, values)
])

provider = LocalSeriesProvider(series_frame)
series_data = provider.fetch(catalog, start='2024-01-01')
series_data.head()

## 3. Diagnosticar regimen actual

In [ ]:
research = MacroLiquidityResearch(min_periods=3)
result = research.analyze_current(series_data)

result.current

In [ ]:
result.scores.tail()

## 4. Definir forecast in-house

El forecast es independiente del diagnostico actual. Aqui entra la vision de casa: modelos, nowcasts, juicio interno o escenarios.

In [ ]:
forecast = RegimeForecast(
    scenarios=(
        ForecastScenario(
            name='base',
            macro_regime='desaceleracion_ordenada',
            liquidity_regime='restrictiva',
            probability=0.60,
            horizon_months=6,
            confidence=0.65,
            rationale='Crecimiento se modera y la liquidez se vuelve menos favorable.',
        ),
        ForecastScenario(
            name='upside',
            macro_regime='expansion_desinflacionaria',
            liquidity_regime='neutral',
            probability=0.25,
            horizon_months=6,
            confidence=0.50,
            rationale='Desinflacion ordenada con condiciones financieras estables.',
        ),
        ForecastScenario(
            name='downside',
            macro_regime='estanflacion',
            liquidity_regime='estresada',
            probability=0.15,
            horizon_months=6,
            confidence=0.45,
            rationale='Inflacion persistente y deterioro de fondeo.',
        ),
    )
)

forecast.scenario_table()

## 5. Traducir regimen actual + esperado a vista de activos

In [ ]:
asset_view = build_asset_class_view(
    current=result.current,
    forecast=forecast,
    current_weight=0.40,
    expected_weight=0.60,
)

asset_view.to_frame()

## 6. Aplicar politica del cliente y generar SelectionContext

In [ ]:
client_policy = mexican_moderate_aggressive_growth_policy()

selection_context = build_selection_context(
    current=result.current,
    forecast=forecast,
    asset_view=asset_view,
    policy=client_policy,
)

selection_context.to_frame()

## 7. Handoff a selection

Usar `selection_context` para filtrar universo, ajustar pesos de scoring y documentar por que el research favorece ciertos estilos o clases de activo.

In [ ]:
selection_context.asset_class_overweights, selection_context.preferred_styles, selection_context.score_tilts